<a href="https://colab.research.google.com/github/tusharastirmind-commits/Info5731_Python/blob/main/Shrivastava_Tushar_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:
# Cell 1 - Install
!pip install requests beautifulsoup4

In [ ]:
!wget -q -O /tmp/chrome.deb https://mirror.cs.uchicago.edu/google-chrome/pool/main/g/google-chrome-stable/google-chrome-stable_114.0.5735.90-1_amd64.deb
!apt-get install -y /tmp/chrome.deb -q
!pip install selenium==4.10.0 webdriver-manager beautifulsoup4 pandas -q

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libu2f-udev libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libu2f-udev
  libvulkan1 libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 13 newly installed, 0 to remove and 104 not upgraded.
Need to get 11.2 MB/105 MB of archives.
After this operation, 367 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/

In [ ]:
!pip install requests pandas -q

import requests
import pandas as pd
import time
import random
import json

MOVIES = [
    {"title": "Dune: Part Two",       "id": "tt15239678"},
    {"title": "Inside Out 2",         "id": "tt22022452"},
    {"title": "Deadpool & Wolverine", "id": "tt6263850"},
    {"title": "Moana 2",              "id": "tt11220008"},
]

TARGET     = 1000
OUTPUT_CSV = "imdb_reviews.csv"

HEADERS = {
    "User-Agent"  : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept"      : "application/json",
    "Content-Type": "application/json",
    "Referer"     : "https://www.imdb.com/",
}

session = requests.Session()
session.headers.update(HEADERS)


def fetch_reviews(movie_id, after_cursor=None):
    after_part = f', after: "{after_cursor}"' if after_cursor else ""

    query = {
        "query": f"""
        {{
          title(id: "{movie_id}") {{
            reviews(first: 25{after_part}) {{
              edges {{
                node {{
                  author {{ nickName }}
                  summary {{ originalText }}
                  text {{ originalText {{ plainText }} }}
                  authorRating
                  submissionDate
                  helpfulness {{ upVotes downVotes }}
                }}
              }}
              pageInfo {{
                hasNextPage
                endCursor
              }}
            }}
          }}
        }}
        """
    }

    try:
        resp = session.post(
            "https://api.graphql.imdb.com/",
            json=query,
            timeout=30
        )
        print(f"  Status: {resp.status_code}")
        if resp.status_code != 200:
            return [], None, False
        data = resp.json()
        reviews_data = data["data"]["title"]["reviews"]
        edges        = reviews_data["edges"]
        page_info    = reviews_data["pageInfo"]
        return edges, page_info["endCursor"], page_info["hasNextPage"]
    except Exception as e:
        print(f"  Error: {e}")
        return [], None, False


def parse_edges(edges, movie_title, movie_id):
    reviews = []
    for edge in edges:
        node   = edge["node"]
        body   = node.get("text", {}).get("originalText", {}).get("plainText", "")
        title  = node.get("summary", {}).get("originalText", "")
        author = node.get("author", {}).get("nickName", "Anonymous")
        rating = node.get("authorRating", "N/A")
        date   = node.get("submissionDate", "")

        if body:
            reviews.append({
                "movie_title": movie_title,
                "movie_id"   : movie_id,
                "title"      : title,
                "rating"     : rating,
                "author"     : author,
                "date"       : date,
                "review"     : body,
            })
    return reviews


def scrape_movie(movie, max_reviews=350):
    all_reviews = []
    seen        = set()
    cursor      = None
    page        = 1

    print(f"\nScraping: {movie['title']}")

    while len(all_reviews) < max_reviews:
        edges, cursor, has_next = fetch_reviews(movie["id"], cursor)

        reviews = parse_edges(edges, movie["title"], movie["id"])
        for r in reviews:
            key = r["review"][:80]
            if key not in seen:
                seen.add(key)
                all_reviews.append(r)

        print(f"  Page {page}, total: {len(all_reviews)}")

        if not has_next or not cursor:
            print(f"  No more pages.")
            break

        page += 1
        time.sleep(random.uniform(1, 2))

    return all_reviews[:max_reviews]


all_data = []

for movie in MOVIES:
    reviews = scrape_movie(movie, max_reviews=350)
    all_data.extend(reviews)
    print(f"Running total: {len(all_data)}")

    if len(all_data) >= TARGET:
        break

    time.sleep(random.uniform(2, 4))

if not all_data:
    print("No data collected.")
else:
    df = pd.DataFrame(all_data)
    df.drop_duplicates(subset=["review"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"\nSaved {len(df)} reviews to '{OUTPUT_CSV}'")
    print(df["movie_title"].value_counts().to_string())


Scraping: Dune: Part Two
  Status: 200
  Page 1, total: 25
  Status: 200
  Page 2, total: 50
  Status: 200
  Page 3, total: 75
  Status: 200
  Page 4, total: 100
  Status: 200
  Page 5, total: 125
  Status: 200
  Page 6, total: 150
  Status: 200
  Page 7, total: 175
  Status: 200
  Page 8, total: 199
  Status: 200
  Page 9, total: 224
  Status: 200
  Page 10, total: 249
  Status: 200
  Page 11, total: 274
  Status: 200
  Page 12, total: 299
  Status: 200
  Page 13, total: 324
  Status: 200
  Page 14, total: 349
  Status: 200
  Page 15, total: 374
Running total: 350

Scraping: Inside Out 2
  Status: 200
  Page 1, total: 25
  Status: 200
  Page 2, total: 50
  Status: 200
  Page 3, total: 75
  Status: 200
  Page 4, total: 100
  Status: 200
  Page 5, total: 125
  Status: 200
  Page 6, total: 150
  Status: 200
  Page 7, total: 175
  Status: 200
  Page 8, total: 200
  Status: 200
  Page 9, total: 225
  Status: 200
  Page 10, total: 250
  Status: 200
  Page 11, total: 275
  Status: 200
  Pag

In [ ]:
resp = session.get("https://www.imdb.com/title/tt15239678/reviews", timeout=30)
print("Status:", resp.status_code)
print("Title:", BeautifulSoup(resp.text, "html.parser").title.text if BeautifulSoup(resp.text, "html.parser").title else "No title")
print(resp.text[:2000])

Status: 200
Title: Dune: Part Two (2024) - User reviews - IMDb
<!DOCTYPE html><html lang="en-US" xmlns:og="http://opengraphprotocol.org/schema/" xmlns:fb="http://www.facebook.com/2008/fbml"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><script>if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }</script><script>window.addEventListener('load', (event) => {
        if (typeof window.csa !== 'undefined' && typeof window.csa === 'function') {
            var csaLatencyPlugin = window.csa('Content', {
                element: {
                    slotId: 'LoadTitle',
                    type: 'service-call'
                }
            });
            csaLatencyPlugin('mark', 'clickToBodyBegin', 1772442534903);
        }
    })</script><title>Dune: Part Two (2024) - User reviews - IMDb</title><meta name="description" content="Dune: Part Two (2024) - Movies, TV, Celebs, and more..." data-id="main"/><meta name="keywords" content="Reviews, Show

# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
!pip install nltk pandas -q

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

df = pd.read_csv("imdb_reviews.csv")
print(f"Loaded {len(df)} rows")

df.dropna(subset=["review"], inplace=True)
df.reset_index(drop=True, inplace=True)


# (1) Remove special characters and punctuation
def remove_special_chars(text):
    return re.sub(r"[^a-zA-Z\s]", " ", str(text))

df["step1_no_special"] = df["review"].apply(remove_special_chars)
print("Step 1 done - special characters removed")
print(df["step1_no_special"].head(2).values)


# (2) Remove numbers
def remove_numbers(text):
    return re.sub(r"\d+", "", text)

df["step2_no_numbers"] = df["step1_no_special"].apply(remove_numbers)
print("\nStep 2 done - numbers removed")
print(df["step2_no_numbers"].head(2).values)


# (3) Remove stopwords
STOP_WORDS = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    return " ".join(t for t in tokens if t.lower() not in STOP_WORDS)

df["step3_no_stopwords"] = df["step2_no_numbers"].apply(remove_stopwords)
print("\nStep 3 done - stopwords removed")
print(df["step3_no_stopwords"].head(2).values)


# (4) Lowercase
df["step4_lowercase"] = df["step3_no_stopwords"].str.lower()
print("\nStep 4 done - lowercased")
print(df["step4_lowercase"].head(2).values)


# (5) Stemming
stemmer = PorterStemmer()

def stem_text(text):
    tokens = word_tokenize(text)
    return " ".join(stemmer.stem(t) for t in tokens)

df["step5_stemmed"] = df["step4_lowercase"].apply(stem_text)
print("\nStep 5 done - stemming applied")
print(df["step5_stemmed"].head(2).values)


# (6) Lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(text)
    return " ".join(lemmatizer.lemmatize(t) for t in tokens)

df["clean_text"] = df["step4_lowercase"].apply(lemmatize_text)
print("\nStep 6 done - lemmatization applied")
print(df["clean_text"].head(2).values)


df.to_csv("imdb_reviews_cleaned.csv", index=False, encoding="utf-8")
print(f"\nSaved cleaned data to 'imdb_reviews_cleaned.csv'")
print(df[["review", "clean_text"]].head(3))

Loaded 1050 rows
Step 1 done - special characters removed
['I m going to write this as a review for both Dune movies  so I ll include my thoughts about Dune part one throughout  For me  the most difficult part about rating these movies is trying to judge them as they stand on their own without comparing them to the original book \n\nAs movies  I think both are great  the cinematography is amazing  the sound design is good  the acting is good  I liked the visual style and how they interpreted the various technologies of the world of Dune  The soundtrack is good  it can be a little overbearing at times but the music in part two felt like an improvement on this from part one  One of the biggest complaints people had with both movies is the pacing  saying that the first movie was dragging and the second was rushing  I agree with the sentiments about the latter  but personally I enjoyed the slower pacing of the first Dune movie  The second movie definitely moves quickly through its first ha

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
!pip install nltk spacy pandas -q
!python -m spacy download en_core_web_sm -q

import pandas as pd
import nltk
import spacy
from nltk import pos_tag, word_tokenize, ne_chunk
from collections import Counter

nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("maxent_ne_chunker", quiet=True)
nltk.download("maxent_ne_chunker_tab", quiet=True)
nltk.download("words", quiet=True)

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("imdb_reviews_cleaned.csv")
df.dropna(subset=["clean_text"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Loaded {len(df)} rows")

all_texts = df["clean_text"].astype(str).tolist()


# (1) POS Tagging on full data
print("\n===== POS TAGGING =====")

noun_count = 0
verb_count = 0
adj_count  = 0
adv_count  = 0

for i, text in enumerate(all_texts):
    tokens = word_tokenize(text)
    tags   = pos_tag(tokens)
    for word, tag in tags:
        if tag.startswith("NN"):   noun_count += 1
        elif tag.startswith("VB"): verb_count += 1
        elif tag.startswith("JJ"): adj_count  += 1
        elif tag.startswith("RB"): adv_count  += 1

    if (i + 1) % 100 == 0:
        print(f"  POS processed {i+1}/{len(all_texts)} reviews...")

print(f"\nTotal Nouns      : {noun_count}")
print(f"Total Verbs      : {verb_count}")
print(f"Total Adjectives : {adj_count}")
print(f"Total Adverbs    : {adv_count}")

example_tokens = word_tokenize(all_texts[0])
example_tags   = pos_tag(example_tokens)
print("\nPOS tags for first review (first 20 words):")
print(example_tags[:20])


# (2) Constituency and Dependency Parsing
print("\n===== CONSTITUENCY AND DEPENDENCY PARSING =====")

print("Constituency Parse Trees (all reviews, showing first 5 output):")
trees_printed = 0
for i, text in enumerate(all_texts):
    tokens = word_tokenize(text[:300])
    tags   = pos_tag(tokens)
    tree   = ne_chunk(tags)
    if trees_printed < 5:
        print(f"\nReview {i+1} constituency tree:")
        print(tree)
        trees_printed += 1

print("\nDependency Parsing (all reviews, showing first 5 output):")
dep_printed = 0
for doc in nlp.pipe(all_texts, batch_size=50):
    if dep_printed < 5:
        print(f"\nReview {dep_printed+1} dependencies:")
        for token in doc:
            print(f"  {token.text:15} -> dep: {token.dep_:10} -> head: {token.head.text}")
        dep_printed += 1

print("\nExample explanation using first review:")
example_doc = nlp(all_texts[0][:300])
print(f"Sentence: {all_texts[0][:200]}")
print("\nConstituency tree groups words into nested phrases like NP (noun phrase) and VP (verb phrase).")
print("Dependency tree shows the grammatical relationship between each word and its head word.")
print("\nDependency breakdown:")
for token in example_doc:
    print(f"  '{token.text}' --[{token.dep_}]--> '{token.head.text}'")


# (3) Named Entity Recognition on full data
print("\n===== NAMED ENTITY RECOGNITION =====")

entity_counts   = Counter()
entity_examples = {}

total = len(all_texts)
for i, doc in enumerate(nlp.pipe(all_texts, batch_size=50)):
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG", "GPE", "PRODUCT", "DATE"]:
            entity_counts[ent.label_] += 1
            if ent.label_ not in entity_examples:
                entity_examples[ent.label_] = ent.text

    if (i + 1) % 100 == 0:
        print(f"  NER processed {i+1}/{total} reviews...")

print(f"\nEntity counts across all {total} reviews:")
for label, count in entity_counts.most_common():
    print(f"  {label:10}: {count:5}  (example: {entity_examples.get(label, 'N/A')})")

print("\nDetailed NER for first 3 reviews:")
for i, text in enumerate(all_texts[:3]):
    doc      = nlp(text[:500])
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"\nReview {i+1}: {entities}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loaded 1050 rows

===== POS TAGGING =====
  POS processed 100/1050 reviews...
  POS processed 200/1050 reviews...
  POS processed 300/1050 reviews...
  POS processed 400/1050 reviews...
  POS processed 500/1050 reviews...
  POS processed 600/1050 reviews...
  POS processed 700/1050 reviews...
  POS processed 800/1050 reviews...
  POS processed 900/1050 reviews...
  POS processed 1000/1050 reviews...

Total Nouns      : 56446
Total Verbs      : 23202
Total Adjectives : 27118
Total Adverbs    : 11883

POS tags for first review (first 20 words):
[('going', 'VBG'), ('write', 'JJ')

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [ ]:
import requests
import pandas as pd
import time

GITHUB_TOKEN = ""

HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

BASE_URL   = "https://api.github.com/search/repositories"
OUTPUT_CSV = "github_marketplace_actions.csv"

SEARCH_QUERIES = [
    "topic:github-actions",
    "topic:actions marketplace",
    "topic:github-action stars:>10",
]

records  = []
seen_ids = set()

for query in SEARCH_QUERIES:
    print(f"Query: '{query}'")

    for page in range(1, 11):
        params = {
            "q"       : query,
            "sort"    : "stars",
            "order"   : "desc",
            "per_page": 100,
            "page"    : page,
        }

        try:
            resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=20)

            if resp.status_code == 403:
                reset_time = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
                wait = max(reset_time - int(time.time()), 10)
                print(f"  Rate limited, waiting {wait}s")
                time.sleep(wait)
                resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=20)

            resp.raise_for_status()
            items = resp.json().get("items", [])

            if not items:
                print(f"  No more results at page {page}")
                break

            for repo in items:
                rid = repo.get("id")
                if rid in seen_ids:
                    continue
                seen_ids.add(rid)
                records.append({
                    "product_name": repo.get("name", ""),
                    "full_name"   : repo.get("full_name", ""),
                    "description" : repo.get("description", "") or "",
                    "url"         : repo.get("html_url", ""),
                    "stars"       : repo.get("stargazers_count", 0),
                    "forks"       : repo.get("forks_count", 0),
                    "language"    : repo.get("language", "") or "",
                    "topics"      : ", ".join(repo.get("topics", [])),
                    "created_at"  : repo.get("created_at", ""),
                    "updated_at"  : repo.get("updated_at", ""),
                    "page"        : page,
                })

            print(f"  Page {page}, total collected: {len(records)}")
            time.sleep(2)

        except Exception as e:
            print(f"  Error: {e}")
            break

    if len(records) >= 1000:
        break

df = pd.DataFrame(records[:1000])
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"\nSaved {len(df)} records to '{OUTPUT_CSV}'")
print(df[["product_name", "description", "url"]].head(5).to_string(index=False))

Query: 'topic:github-actions'
  Page 1, total collected: 100
  Page 2, total collected: 200
  Page 3, total collected: 300
  Page 4, total collected: 400
  Page 5, total collected: 500
  Page 6, total collected: 600
  Page 7, total collected: 700
  Page 8, total collected: 800
  Page 9, total collected: 900
  Page 10, total collected: 1000

Saved 1000 records to 'github_marketplace_actions.csv'
   product_name                                                                                                                                                             description                                      url
            act                                                                                                                                       Run your GitHub Actions locally 🚀            https://github.com/nektos/act
          gitea Git with a cup of tea! Painless self-hosted all-in-one software development service, including Git hosting, code review, team collaborat

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

df = pd.read_csv("github_marketplace_actions.csv")
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")

STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text   = re.sub(r"<[^>]+>", " ", str(text))
    text   = re.sub(r"[^a-zA-Z\s]", " ", text)
    text   = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return " ".join(tokens)

df["clean_description"]  = df["description"].apply(preprocess_text)
df["clean_product_name"] = df["product_name"].apply(preprocess_text)

print("\nPreprocessing done.")
print(df[["product_name", "description", "clean_description"]].head(5))


print("\n===== DATA QUALITY CHECKS =====")

print(f"\nTotal records: {len(df)}")

print("\n1. Missing values per column:")
print(df.isnull().sum())

print("\n2. Empty strings per column:")
for col in df.columns:
    empty = (df[col].astype(str).str.strip() == "").sum()
    print(f"  {col:25}: {empty}")

before = len(df)
df.drop_duplicates(subset=["url"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"\n3. Duplicates removed: {before - len(df)}")
print(f"   Records after dedup: {len(df)}")

df["description"].fillna("No description available", inplace=True)
df["clean_description"].replace("", "no description available", inplace=True)
print(f"\n4. Missing descriptions filled.")

print("\n5. Data types:")
print(df.dtypes)

df["stars"] = pd.to_numeric(df["stars"], errors="coerce").fillna(0).astype(int)
print(f"\n6. Stars range: min={df['stars'].min()}, max={df['stars'].max()}")

df["desc_length"] = df["description"].apply(lambda x: len(str(x).split()))
print(f"\n7. Description length stats:")
print(df["desc_length"].describe())
print(f"   Very short descriptions (<3 words): {(df['desc_length'] < 3).sum()}")

df.to_csv("github_marketplace_actions_cleaned.csv", index=False, encoding="utf-8")
print(f"\nSaved cleaned data to 'github_marketplace_actions_cleaned.csv'")
print(f"Final shape: {df.shape}")

Loaded 1000 rows
Columns: ['product_name', 'full_name', 'description', 'url', 'stars', 'forks', 'language', 'topics', 'created_at', 'updated_at', 'page']

Preprocessing done.
      product_name                                        description  \
0              act                  Run your GitHub Actions locally 🚀   
1            gitea  Git with a cup of tea! Painless self-hosted al...   
2  awesome-actions  A curated list of awesome actions to use on Gi...   
3       goreleaser                    Release engineering, simplified   
4         ubicloud  Open source alternative to AWS. Elastic comput...   

                                   clean_description  
0                          run github action locally  
1  git cup tea painless self hosted one software ...  
2             curated list awesome action use github  
3                     release engineering simplified  
4  open source alternative aws elastic compute bl...  

===== DATA QUALITY CHECKS =====

Total records: 1000

1

/tmp/ipython-input-1124/1151774872.py:56: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["description"].fillna("No description available", inplace=True)
/tmp/ipython-input-1124/1151774872.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [ ]:
!pip install Mastodon.py

In [ ]:
# Question 5: Collect ML/AI posts from Mastodon + Full Cleaning Pipeline

from mastodon import Mastodon
import pandas as pd
import re
import time
import nltk
from nltk.corpus   import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer

# Download NLTK resources
for r in ["punkt", "stopwords", "wordnet", "omw-1.4", "punkt_tab"]:
    nltk.download(r, quiet=True)


# PART 1: DATA COLLECTION

ACCESS_TOKEN  = ""
MASTODON_HOST = "https://mastodon.social"
OUTPUT_CSV    = "mastodon_ml_ai_posts.csv"

# Init Mastodon client
mastodon = Mastodon(
    access_token = ACCESS_TOKEN,
    api_base_url = MASTODON_HOST,
)

HASHTAGS      = ["machinelearning", "artificialintelligence",
                 "datascience", "deeplearning", "nlp"]
POSTS_PER_TAG = 200      # 200 × 5 = 1000 posts total

records  = []
seen_ids = set()

print(" Starting Mastodon data collection ...\n")

for hashtag in HASHTAGS:
    collected = 0
    max_id    = None
    print(f" Fetching #{hashtag} ...")

    while collected < POSTS_PER_TAG:
        try:
            kwargs = {"limit": 40}
            if max_id:
                kwargs["max_id"] = max_id

            posts = mastodon.timeline_hashtag(hashtag, **kwargs)

            if not posts:
                print(f"  No more posts for #{hashtag}")
                break

            for post in posts:
                pid = post["id"]
                if pid in seen_ids:
                    continue
                seen_ids.add(pid)

                # Strip HTML tags from content
                content_raw  = post.get("content", "")
                content_text = re.sub(r"<[^>]+>", " ", content_raw).strip()
                content_text = re.sub(r"\s+", " ", content_text)

                if len(content_text) < 20:
                    continue

                account = post.get("account", {})
                records.append({
                    "post_id"       : str(pid),
                    "username"      : account.get("acct", ""),
                    "display_name"  : account.get("display_name", ""),
                    "text"          : content_text,
                    "hashtag"       : hashtag,
                    "created_at"    : post.get("created_at", ""),
                    "reblogs_count" : post.get("reblogs_count", 0),
                    "favourites"    : post.get("favourites_count", 0),
                    "language"      : post.get("language", ""),
                    "url"           : post.get("url", ""),
                })
                collected += 1

                if collected >= POSTS_PER_TAG:
                    break

            max_id = posts[-1]["id"]
            print(f"  #{hashtag} → {collected}/{POSTS_PER_TAG} collected ...")
            time.sleep(1)

        except Exception as e:
            print(f"  ✗ Error: {e}")
            time.sleep(10)
            break

    print(f"   #{hashtag} done — {collected} posts\n")

# Save raw data
df = pd.DataFrame(records)
df.drop_duplicates(subset="post_id", inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"  Raw data saved → '{OUTPUT_CSV}'  ({len(df)} posts)")
print(df[["username", "hashtag", "text"]].head(3))

# PART 2: DATA CLEANING + QUALITY CHECK

print("\n" + "=" * 60)
print("PART 2: CLEANING & QUALITY CHECK")
print("=" * 60)

STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Cleaning functions
def remove_urls(text):
    """(1) Remove URLs"""
    return re.sub(r"http\S+|www\.\S+", " ", str(text))

def remove_mentions_hashtags(text):
    """(2) Remove @mentions and #hashtags"""
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    return text

def remove_special(text):
    """(3) Remove special characters & punctuation"""
    return re.sub(r"[^a-zA-Z\s]", " ", text)

def remove_extra_spaces(text):
    """(4) Remove extra whitespace"""
    return re.sub(r"\s+", " ", text).strip()

def to_lowercase(text):
    """(5) Lowercase"""
    return text.lower()

def remove_stopwords(text):
    """(6) Remove stopwords"""
    tokens = word_tokenize(text)
    return " ".join(t for t in tokens if t not in STOP_WORDS and len(t) > 2)

def lemmatize(text):
    """(7) Lemmatize"""
    tokens = word_tokenize(text)
    return " ".join(lemmatizer.lemmatize(t) for t in tokens)

# Apply each step

df["step1_no_urls"]      = df["text"].apply(remove_urls)
print("  ✓ Step 1: URLs removed")

df["step2_no_mentions"]  = df["step1_no_urls"].apply(remove_mentions_hashtags)
print("  ✓ Step 2: Mentions & hashtags removed")

df["step3_no_special"]   = df["step2_no_mentions"].apply(remove_special)
print("  ✓ Step 3: Special characters removed")

df["step4_no_spaces"]    = df["step3_no_special"].apply(remove_extra_spaces)
print("  ✓ Step 4: Extra whitespace removed")

df["step5_lowercase"]    = df["step4_no_spaces"].apply(to_lowercase)
print("  ✓ Step 5: Lowercased")

df["step6_no_stopwords"] = df["step5_lowercase"].apply(remove_stopwords)
print("  ✓ Step 6: Stopwords removed")

df["clean_text"]         = df["step6_no_stopwords"].apply(lemmatize)
print("  ✓ Step 7: Lemmatization done")

print("\n  Sample — original vs cleaned:")
print(df[["text", "clean_text"]].head(3).to_string())

# Data Quality Report
print(f"  Shape                : {df.shape}")
print(f"\n  Missing values:\n{df[['post_id','username','text','clean_text']].isnull().sum()}")

dupes = df.duplicated(subset="post_id").sum()
print(f"\n  Duplicate post_ids   : {dupes}")
df.drop_duplicates(subset="post_id", inplace=True)

empty = (df["clean_text"].str.strip() == "").sum()
print(f"  Empty clean_text     : {empty}")
df = df[df["clean_text"].str.strip() != ""]

print(f"\n  Language distribution:\n{df['language'].value_counts().head(5)}")
print(f"\n  Posts per hashtag:\n{df['hashtag'].value_counts()}")

df["word_count"] = df["clean_text"].apply(lambda x: len(str(x).split()))
print(f"\n  Avg words per post   : {df['word_count'].mean():.1f}")
print(f"  Min / Max words      : {df['word_count'].min()} / {df['word_count'].max()}")

#  Save final clean file
df.reset_index(drop=True, inplace=True)
df.to_csv("mastodon_ml_ai_clean.csv", index=False, encoding="utf-8")
print(f"\n  Final clean file saved → 'mastodon_ml_ai_clean.csv'  ({len(df)} rows)")
print(df[["post_id", "username", "hashtag", "clean_text"]].head(5))

 Starting Mastodon data collection ...

 Fetching #machinelearning ...
  #machinelearning → 40/200 collected ...
  #machinelearning → 80/200 collected ...
  #machinelearning → 120/200 collected ...
  #machinelearning → 160/200 collected ...
  #machinelearning → 200/200 collected ...
   #machinelearning done — 200 posts

 Fetching #artificialintelligence ...
  #artificialintelligence → 38/200 collected ...
  #artificialintelligence → 78/200 collected ...
  #artificialintelligence → 117/200 collected ...
  #artificialintelligence → 157/200 collected ...
  #artificialintelligence → 197/200 collected ...
  #artificialintelligence → 200/200 collected ...
   #artificialintelligence done — 200 posts

 Fetching #datascience ...
  #datascience → 38/200 collected ...
  #datascience → 78/200 collected ...
  #datascience → 118/200 collected ...
  #datascience → 158/200 collected ...
  #datascience → 198/200 collected ...
  #datascience → 200/200 collected ...
   #datascience done — 200 posts

 Fet

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

I really enjoyed this assignment. It really widened my horizin regarding python usage in scrapping. The part i enjoyed most is Web Scrape tweets from Twitter using the Tweepy API.Since tweeter is now paid, Therefore i used matrodon api. It is exactly like tweeter and reddit. i craeted account and got api details from it and used it to fetch that. This was a really hands on assignment. I performed various operations like cleaning and quality checks. I learnt a lot from this.